# Parte 1 - Taller Practico y Conceptual de PDI
## Modulo A: Analisis y Ecualizacion de Histogramas (7.5%)

Universidad de Antioquia - Procesamiento Digital de Imagenes - 2026-II

En este modulo se trabaja con histogramas (PDF discreta de intensidades) y se compara la ecualizacion global con la adaptativa CLAHE. Imagen de trabajo: `im1.png` (incluida junto al notebook).


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

def mostrar(img, titulo, cmap=None):
    plt.figure(figsize=(7, 5))
    if img.ndim == 3:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap=cmap or 'gray', vmin=0, vmax=255)
    plt.title(titulo)
    plt.axis('off')
    plt.show()

def histograma(img, canal=0, mascara=None):
    return cv2.calcHist([img], [canal], mascara, [256], [0, 256]).flatten()

img = cv2.imread('im1.png')
assert img is not None, 'No se encontro im1.png'
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
print('Imagen:', img.shape, '| OpenCV', cv2.__version__)

---
## Ejercicio A.1 - Calculo de Histogramas

El histograma es la **funcion de densidad de probabilidad discreta** de las intensidades de los pixeles:

$$
h(r_k) = n_k
$$

donde $n_k$ es el numero de pixeles con intensidad $r_k$. Normalizado por el total $N$:

$$
p(r_k) = \frac{n_k}{N}
$$

Se calcula para escala de grises y para cada canal en RGB y HSV.


In [ ]:
mostrar(img, 'Imagen original')

plt.figure(figsize=(8, 3))
plt.plot(histograma(gray), color='k')
plt.title('Histograma en escala de grises')
plt.xlabel('Intensidad'); plt.ylabel('N. pixeles'); plt.xlim(0, 255)
plt.grid(alpha=0.3); plt.show()

print(f'Media: {gray.mean():.1f} | Desv. estandar: {gray.std():.1f}')
print(f'Pixeles oscuros (<64): {100*np.mean(gray < 64):.1f}%')

# Lectura: la masa esta en intensidades bajas -> imagen oscura, contraste concentrado a la izquierda.

In [ ]:
plt.figure(figsize=(8, 3))
for i, c, n in zip(range(3), ('b', 'g', 'r'), ('B', 'G', 'R')):
    plt.plot(histograma(img, i), color=c, label=n)
plt.title('Histogramas por canal - RGB'); plt.xlabel('Intensidad'); plt.ylabel('N. pixeles')
plt.xlim(0, 255); plt.legend(); plt.grid(alpha=0.3); plt.show()

plt.figure(figsize=(8, 3))
for i, c, n in zip(range(3), ('m', 'c', 'y'), ('H (Tono)', 'S (Sat.)', 'V (Brillo)')):
    plt.plot(histograma(hsv, i), color=c, label=n)
plt.title('Histogramas por canal - HSV'); plt.xlabel('Valor'); plt.ylabel('N. pixeles')
plt.xlim(0, 255); plt.legend(); plt.grid(alpha=0.3); plt.show()

# Lectura: R, G, B con medias casi iguales -> escena acromatica (grises/marrones).
# En HSV, S baja -> poca saturacion; V repite el patron de grises -> escena oscura.

---
## Ejercicio A.2 - Ecualizacion Global vs CLAHE

**Ecualizacion global** (`cv2.equalizeHist`): aplica una sola CDF a toda la imagen, donde la transformacion es:

$$
s = (L-1) \cdot \mathrm{CDF}(r)
$$

con $L = 256$ niveles. La pendiente de la transformacion es:

$$
\frac{ds}{dr} \propto p(r)
$$

Esto significa que donde el histograma tiene un pico (muchos pixeles en pocas intensidades), la pendiente es **enorme** y cualquier ruido de entrada se amplifica a la salida.

**CLAHE** (`cv2.createCLAHE`): divide la imagen en celdas (`tileGridSize`) y ecualiza cada una con su propia CDF local, recortando el histograma segun `clipLimit` para acotar la ganancia maxima. Se prueba con `clipLimit` 2.0 y 4.0, y `tileGridSize` (8,8) y (16,16).


In [ ]:
eq_global = cv2.equalizeHist(gray)

configs = [(2.0, (8, 8)), (2.0, (16, 16)), (4.0, (8, 8)), (4.0, (16, 16))]
clahe_results = {}
for clip, tile in configs:
    c = cv2.createCLAHE(clipLimit=clip, tileGridSize=tile).apply(gray)
    clahe_results[f'clip={clip}, tile={tile}'] = c

imgs = [gray, eq_global] + list(clahe_results.values())
titulos = ['Original', 'Eq. global'] + list(clahe_results.keys())

plt.figure(figsize=(14, 8))
for i, (im, t) in enumerate(zip(imgs, titulos)):
    plt.subplot(2, 3, i + 1)
    plt.imshow(im, cmap='gray', vmin=0, vmax=255)
    plt.title(t, fontsize=9); plt.axis('off')
plt.tight_layout(); plt.show()

# --- Comparacion de histogramas en dos plots para que se distingan bien ---
# Plot 1: Original vs Ecualizacion global (2 lineas, comparacion directa)
plt.figure(figsize=(13, 4))

plt.subplot(1, 2, 1)
plt.plot(histograma(gray), color='black', lw=2.5, label='Original')
plt.plot(histograma(eq_global), color='red', lw=2, ls='--', label='Eq. global')
plt.title('Original vs Ecualizacion global')
plt.xlabel('Intensidad'); plt.ylabel('N. pixeles')
plt.xlim(0, 255); plt.legend(); plt.grid(alpha=0.3)

# Plot 2: Original vs 4 configuraciones de CLAHE
# Color -> familia de clipLimit (azul=2.0, naranja=4.0)
# Estilo de linea -> tileGridSize (solida=8x8, discontinua=16x16)
plt.subplot(1, 2, 2)
plt.plot(histograma(gray), color='black', lw=2.5, label='Original')
plt.plot(histograma(clahe_results['clip=2.0, tile=(8, 8)']),
         color='tab:blue', lw=2, ls='-', label='CLAHE c=2.0 t=(8,8)')
plt.plot(histograma(clahe_results['clip=2.0, tile=(16, 16)']),
         color='tab:blue', lw=2, ls='--', label='CLAHE c=2.0 t=(16,16)')
plt.plot(histograma(clahe_results['clip=4.0, tile=(8, 8)']),
         color='tab:orange', lw=2, ls='-', label='CLAHE c=4.0 t=(8,8)')
plt.plot(histograma(clahe_results['clip=4.0, tile=(16, 16)']),
         color='tab:orange', lw=2, ls='--', label='CLAHE c=4.0 t=(16,16)')
plt.title('Original vs 4 configuraciones CLAHE')
plt.xlabel('Intensidad'); plt.ylabel('N. pixeles')
plt.xlim(0, 255); plt.legend(fontsize=8, loc='upper right'); plt.grid(alpha=0.3)

plt.tight_layout(); plt.show()

# Lectura: la eq. global extiende el histograma a todo [0,255] pero granula el fondo.
# CLAHE con clipLimit bajo (2.0, azul) realza sin ese ruido.
# CLAHE con clipLimit alto (4.0, naranja) logra mas contraste pero el grano vuelve a notarse.
# Linea solida vs discontinua compara el efecto del tamano de celda.

---
## Pregunta 1.1 - Sobre-amplificacion de ruido

> Por que la ecualizacion global arruina zonas homogeneas (pared lisa, fondo) y CLAHE no?

**Respuesta.** La ecualizacion global aplica una sola CDF a toda la imagen. En una zona homogenea la PDF tiene un **pico muy alto y estrecho**, asi que la CDF presenta un **salto casi vertical** ahi y la pendiente:

$$
\frac{dT}{dr} \propto p(r)
$$

se vuelve enorme. Cualquier pequena diferencia de entrada (ruido del sensor de $\pm 2$ niveles) se proyecta a decenas de niveles de salida, asi que el ruido, invisible antes, aparece como manchas de alto contraste.

CLAHE corrige esto en dos frentes:

1. **Recorte (clipping):** cualquier bin del histograma local que supere `clipLimit` se recorta antes de calcular la CDF. Asi la pendiente maxima de la CDF local queda acotada.
2. **Localidad:** la CDF se estima por celdas (`tileGridSize`), de modo que el pico de una region solo gobierna su propia transformacion y no el resto.

Resultado: contraste local sin sobre-amplificar el fondo liso.


---
## Pregunta 1.2 - Canal de brillo vs RGB

> Por que es mala practica ecualizar R, G y B por separado y es mejor aplicar CLAHE solo sobre V (HSV) o L (CIELAB)?

**Respuesta.** El color percibido depende de las **proporciones** R:G:B, no de sus valores absolutos. Si se ecualiza cada canal con su propia CDF, las razones cambian arbitrariamente y los tonos se destruyen (aparecen colores falsos).

Los espacios HSV y CIELAB **separan** la informacion:

- Un canal de **luminancia** (V en HSV, L en CIELAB) que concentra el brillo.
- Canales de **crominancia** (H, S en HSV; a, b en CIELAB) que concentran el color.

Aplicar CLAHE **solo** sobre V o L redistribuye el brillo sin alterar las proporciones cromaticas, asi el tono se preserva. CIELAB ademas es **perceptualmente uniforme**: variaciones iguales de L corresponden a diferencias de brillo percibido aproximadamente iguales.


In [ ]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

# (a) Mala practica: ecualizar B, G y R por separado
b, g, r = cv2.split(img)
eq_rgb = cv2.merge([cv2.equalizeHist(b), cv2.equalizeHist(g), cv2.equalizeHist(r)])

# (b) Buena practica: CLAHE solo sobre V (HSV)
h, s, v = cv2.split(hsv)
eq_hsv = cv2.cvtColor(cv2.merge([h, s, clahe.apply(v)]), cv2.COLOR_HSV2BGR)

# (c) Buena practica: CLAHE solo sobre L (CIELAB)
lab = cv2.cvtColor(img, cv2.COLOR_BGR2Lab)
L, A, B = cv2.split(lab)
eq_lab = cv2.cvtColor(cv2.merge([clahe.apply(L), A, B]), cv2.COLOR_Lab2BGR)

casos = [img, eq_rgb, eq_hsv, eq_lab]
titulos = ['Original', 'Eq. R,G,B independiente (tonos falsos)', 'CLAHE solo en V (HSV)', 'CLAHE solo en L (CIELAB)']

plt.figure(figsize=(12, 9))
for i, (im, t) in enumerate(zip(casos, titulos)):
    plt.subplot(2, 2, i + 1)
    plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    plt.title(t, fontsize=10); plt.axis('off')
plt.tight_layout(); plt.show()

---
## Conclusiones del Modulo A

1. El histograma es la PDF discreta de las intensidades y permite diagnosticar exposicion y contraste. En `im1.png` la masa en valores bajos indica sombras marcadas.
2. La ecualizacion global (una sola CDF) maximiza el contraste pero su pendiente:

$$
\frac{ds}{dr} \propto p(r)
$$

sobre-amplifica el ruido de zonas homogeneas.

3. CLAHE combina CDF **local** por celdas con **recorte** segun `clipLimit`, asi el contraste se realza sin reventar el fondo liso.
4. En color nunca se ecualizan R, G, B por separado (rompe las proporciones cromaticas); se opera solo sobre la luminancia V o L para preservar el tono.
